In [12]:
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from pathlib import Path

import pandas as pd

In [ ]:
# --- КОНФИГ ---
MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
INPUT_PARQUET = Path("../../data/processed/all_chunks.parquet")
OUTPUT_PARQUET = Path("../../data/processed/all_chunks_with_embeddings.parquet")
BATCH_SIZE = 32

In [ ]:
model = SentenceTransformer(
    MODEL_ID,
    device="cuda", 
    model_kwargs={
        "dtype": torch.bfloat16,
        "attn_implementation": "flash_attention_2",
        "trust_remote_code": True
    },
    tokenizer_kwargs={"padding_side": "left"}
)

In [ ]:
def compute_embeddings(batch):
    embeddings = model.encode(
        batch["text"],
        batch_size=len(batch["text"]),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    
    return {"embedding": embeddings}

In [ ]:
ds = Dataset.from_parquet(str(INPUT_PARQUET))

In [ ]:
print(f"Начинаем векторизацию {len(ds)} чанков...")
ds_with_embeddings = ds.map(
    compute_embeddings,
    batched=True,
    batch_size=BATCH_SIZE,
    desc="Embedding generation"
)

In [ ]:
print(f"Сохранение в {OUTPUT_PARQUET}...")
ds_with_embeddings.to_parquet(str(OUTPUT_PARQUET))
print("Готово.")

In [2]:
INPUT_PARQUET = Path("../../data/processed/all_chunks.parquet")
ds = Dataset.from_parquet(str(INPUT_PARQUET))

In [3]:
ds

Dataset({
    features: ['doc_id', 'text', 'metadata'],
    num_rows: 658473
})

In [13]:
df = pd.read_parquet(INPUT_PARQUET)

In [22]:
df.iloc[3]['text']

'Labels Generated by Large Language Models Help Measure People’s Empathy in Vitro > I Introduction: Empathy computing offers the potential to improve people’s empathic skills, which in turn strengthens interpersonal relationships across various human interactions hasan2024empathy. In healthcare, for example, empathic writing in medical documents (*e.g*.,patient reports) can promote understanding and trust between clinicians and patients jani2012role. Similarly, in education, written communication like emails and feedback on assignments has become a vital medium for expressing care and addressing students’ emotional needs aldrup2022empathy. Journalism also demonstrates the importance of empathy in written narratives. For example, a news article on a family’s recovery after a devastating event often goes beyond factual reporting and offers a compassionate perspective that engages readers emotionally and deepens their connection to the news story. Specifically, this paper measures people’